# BT4012 Group 05 — Colab runner
Runs the heavy steps (5-seed training sweeps, self-training ablation, adversarial injection) on a Colab GPU/CPU with the same pinned dependencies as the repo.
1. Runtime → Change runtime type → GPU (optional; everything also runs on CPU).
2. Fill in the repo URL and, if you need LLM scoring, the API key (never commit it).

In [ ]:
REPO = "https://github.com/<user>/bt4012-adaptive-fraud-detection.git"
!git clone -q $REPO repo && cd repo && pip install -q -r requirements.txt && pip install -q -e .
%cd repo

In [ ]:
# public graph file + checks
!python scripts/01_download.py
# raw text release: upload data/raw/yelpchi_raw/{metadata,reviewContent} via the Files pane or mount Drive
# from google.colab import drive; drive.mount('/content/drive')

In [ ]:
!python scripts/02_build_graph.py --source raw
!python scripts/03_eda.py --source raw

In [ ]:
# LLM scoring is cached in data/llm/*.parquet (committed). Only run this if the cache is incomplete.
# import os; os.environ['ANTHROPIC_API_KEY'] = '...'
# !python scripts/04_llm_score.py --source raw --dry-run

In [ ]:
!python scripts/05_distil_edges.py --source raw
!python scripts/06_train.py --source raw
!python scripts/06_train.py --source raw --weights data/processed/edge_scores_raw.npz --mode soft
!python scripts/06_train.py --source raw --weights data/processed/edge_scores_raw.npz --mode hard --tau 0.5

In [ ]:
!python scripts/07_drift.py --source raw
!python scripts/08_selftrain.py --source raw
!python scripts/08_selftrain.py --source raw --weights data/processed/edge_scores_raw.npz
!python scripts/09_adversarial.py --source raw --recover
!python scripts/10_figures.py

In [ ]:
# bring results back: zip outputs/ and download, or commit outputs/tables + outputs/figures on a branch
!zip -qr outputs.zip outputs/tables outputs/figures outputs/runs && echo done